# TalkNet-ASD: End-to-End Inference Demo
This notebook was created to run the TalkNet-ASD pipeline directly in Google Colab's free GPU tier, fulfilling Task 00007.

**Instructions:**
1. Make sure you are using a GPU runtime: `Runtime -> Change runtime type -> Hardware accelerator: GPU`.
2. Run the cells below sequentially.

In [ ]:
# 1. Setup Environment (Clone Repo & Install Dependencies)
!git clone https://github.com/TaoRuijie/TalkNet-ASD.git
%cd TalkNet-ASD

# Install dependencies (pin scenedetect to 0.5.x as required by TalkNet)
!pip install scenedetect==0.5.6.1
!pip install gdown scipy librosa opencv-python python_speech_features tqdm

# Install ffmpeg (required for audio extraction)
!apt-get update -qq && apt-get install -y -qq ffmpeg

# ---- CRITICAL PATCH ----
# TalkNet-ASD uses deprecated np.int / np.float / np.bool which were removed in NumPy 1.24+.
# We patch the source files in-place to use the Python builtins instead.
!find . -name "*.py" -exec sed -i "s/np\.int)/int)/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.int,/int,/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.int]/int]/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.float)/float)/g" {} +
!find . -name "*.py" -exec sed -i "s/np\.bool)/bool)/g" {} +

print("\n=== Environment setup complete ===")

In [ ]:
# 2. Get a Sample Video
# TalkNet-ASD defaults to reading from the `demo` folder.
!mkdir -p demo
!wget -O demo/sample_demo.mp4 https://github.com/intel-iot-devkit/sample-videos/raw/master/face-demographics-walking-and-pause.mp4

# Check that it downloaded successfully
!ls -lh demo/sample_demo.mp4

In [ ]:
# 3. Run Inference Pipeline
# We pass `--videoName sample_demo` (without the .mp4 extension, as expected by the script).
# The script automatically downloads the pretrained checkpoint from Google Drive on its first run.

!python demoTalkNet.py --videoName sample_demo


In [ ]:
# 4. View Results
# The processed video with bounding boxes and active speaker tags is saved in the `demo/pyavi` folder.
import os

if os.path.exists('demo/sample_demo/pyavi/video_out.avi'):
    print("\n=== SUCCESS ===")
    print("Inference completed! The output video is located at: demo/sample_demo/pyavi/video_out.avi")
    print("You can download this file using the Colab file browser on the left sidebar.")
else:
    !ls -R demo/